In [4]:
import pandas as pd
import numpy as np

# 1. Load & Clean Dataset (from Week 2 rules)
df = pd.read_csv('teaching_prices_messy.csv')
df_clean = df.drop_duplicates().copy()

# Clean text columns
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

# Identify price and item columns
price_col = [c for c in df_clean.columns if 'price' in c.lower()][0]
item_col = [c for c in df_clean.columns if 'item' in c.lower() or 'product' in c.lower() or 'name' in c.lower()][0]

df_clean[price_col] = pd.to_numeric(df_clean[price_col], errors='coerce')
df_valid = df_clean.dropna(subset=[price_col]).copy()

# 2. Compute Mean, Median & First-to-Last % Change per Item
summary_rows = []
for item, group in df_valid.groupby(item_col):
    mean_val = group[price_col].mean()
    median_val = group[price_col].median()
    first_val = group[price_col].iloc[0]
    last_val = group[price_col].iloc[-1]
    pct_change = ((last_val - first_val) / first_val) * 100 if first_val != 0 else np.nan
    
    summary_rows.append({
        'Item': item,
        'Count': len(group),
        'Mean': round(mean_val, 2),
        'Median': round(median_val, 2),
        'First_Price': first_val,
        'Last_Price': last_val,
        'First_to_Last_Pct_Change': round(pct_change, 2)
    })

stats_df = pd.DataFrame(summary_rows)
print("--- DESCRIPTIVE STATISTICS BY ITEM ---")
display(stats_df)

# 3. Impact of Flagged Outlier Value (With vs. Without Flagged Value on Top Item)
sample_item = stats_df.iloc[0]['Item']
item_data = df_valid[df_valid[item_col] == sample_item][price_col]

Q1, Q3 = item_data.quantile(0.25), item_data.quantile(0.75)
IQR = Q3 - Q1
no_outliers = item_data[(item_data >= Q1 - 1.5 * IQR) & (item_data <= Q3 + 1.5 * IQR)]

print(f"\n--- OUTLIER COMPARISON FOR: {sample_item} ---")
print(f"WITH Flagged Values    -> Mean: {item_data.mean():.2f}, Median: {item_data.median():.2f}")
print(f"WITHOUT Flagged Values -> Mean: {no_outliers.mean():.2f}, Median: {no_outliers.median():.2f}")

<ipython-input-4-f3efe8133049>:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_clean.select_dtypes(include=['object']).columns:


--- DESCRIPTIVE STATISTICS BY ITEM ---


,Item,Count,Mean,Median,First_Price,Last_Price,First_to_Last_Pct_Change
0,Oil B,25,316.18,324.00,298.0,350.0,17.45
1,Pulse C,23,205.91,206.00,200.0,213.5,6.75
2,Staple A,22,111.39,112.50,101.0,120.5,19.31
3,Sweetener D,24,576.81,167.75,149.0,184.5,23.83



--- OUTLIER COMPARISON FOR: Oil B ---
WITH Flagged Values    -> Mean: 316.18, Median: 324.00
WITHOUT Flagged Values -> Mean: 324.83, Median: 324.00
